# Confluence — run the experiments (Colab)

Conserved-flow routing for MoE / KGQA. This notebook clones the repo, installs deps,
stages the MetaQA data, and runs the experiments. It auto-uses a GPU if one is attached.

**Recommended:** `Runtime -> Change runtime type -> GPU` (a free T4 is plenty). The MoE-transformer
runs fine on CPU too (~90s/seed); GPU mainly speeds up bigger sweeps and more seeds.

Run the cells top to bottom. Results print inline and figures render at the end.

## 1. Clone the repo (main branch)

In [ ]:
!git clone https://github.com/tanushappapogu-max/Confluence.git
%cd Confluence
!git checkout main
!git log -1 --oneline

## 2. Dependencies + GPU check

In [ ]:
!pip -q install torch numpy matplotlib scipy
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Stage the MetaQA data (OPTIONAL — only for Section 7)
Pulls the KB + 2-hop/3-hop QA from a public mirror. **The headline test (cell 4 / 4b) needs NO data —
you can skip this cell and jump straight to Section 4.** Only run this if you also want the MetaQA
boundary results in Section 7. (1-hop isn't in the mirror and isn't used, so it's skipped.)

In [ ]:
!git clone --depth 1 https://github.com/Tiny-Small/CS5284_Project /tmp/mqa
import os, shutil, glob
os.makedirs('data/metaqa/kb', exist_ok=True)

def stage(patterns, dst, want_biggest=False):
    hits = []
    for pat in patterns:
        hits += glob.glob(f'/tmp/mqa/**/{pat}', recursive=True)
    hits = [h for h in hits if os.path.getsize(h) > 1000]        # skip empty/sample stubs
    if not hits: return None
    src = max(hits, key=os.path.getsize) if want_biggest else min(hits, key=len)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst); return src

# 1-hop is not in this mirror and is unused by the experiments; only 2-hop / 3-hop are needed.
print('kb.txt      <-', stage(['kb.txt'], 'data/metaqa/kb/kb.txt', want_biggest=True))
for h in ['2', '3']:
    for split in ['train', 'test']:
        s = stage([f'{h}-hop/qa_{split}.txt', f'*{h}-hop*/qa_{split}.txt'],
                  f'data/metaqa/{h}-hop/qa_{split}.txt')
        print(f'{h}-hop qa_{split} <-', s)
!echo '--- staged (kb should be 134741 lines, 2-hop train 118980) ---'; wc -l data/metaqa/kb/kb.txt data/metaqa/2-hop/qa_train.txt data/metaqa/2-hop/qa_test.txt

## 4. HEADLINE — MoE transformer: flow router vs top-k gate (real model)
Flow should beat the top-1 gate at **equal compute** (1 expert/layer) with a near-zero illegal rate.
`NSEEDS`/`EPOCHS`/`NTRAIN` are env-overridable; defaults are 3 seeds / 12 epochs / 20k.

In [ ]:
!NSEEDS=3 EPOCHS=12 NTRAIN=20000 python3 moe_transformer.py

### 4b. CONCRETE RESULT — 5-seed variance run (do this on GPU)
Pins down flow's honest mean±std (settles the occasional bad-seed instability), and sweeps the
transformer across legality densities. On a T4 this is a few minutes; these numbers go in the paper.

In [ ]:
# 5-seed headline (honest variance)
!NSEEDS=5 EPOCHS=15 NTRAIN=20000 python3 moe_transformer.py

# density sweep in the real transformer (2 seeds/point)
import subprocess, os
for den in ['0.4','0.6','0.8','1.0']:
    print(f'\n===== density {den} =====')
    env = dict(os.environ, DENSITY=den, NSEEDS='2', EPOCHS='12', NTRAIN='12000')
    out = subprocess.run(['python3','moe_transformer.py'], env=env, capture_output=True, text=True).stdout
    print('\n'.join(l for l in out.splitlines() if 'gate' in l or 'flow' in l or 'density=' in l))

## 5. The coupling condition (synthetic) + MoE-routing comparison
`density_sweep.py`: flow beats independent scoring **iff** steps are coupled (the crossover).
`mycelial_final.py`: flow vs top-k gate vs Routing-Free MoE (4 seeds).
`moe_layer_demo.py`: MoE FFN accuracy/compute frontier.

In [ ]:
!python3 density_sweep.py
!python3 mycelial_final.py
!python3 moe_layer_demo.py

## 6. Scaling: implicit differentiation through the flow fixed point (verification)

In [ ]:
!python3 implicit_diff.py

## 7. MetaQA (real data) — the marginalizable boundary
Needs section 3. The no-flow ablation ties the flow model on 2-hop (KGQA is marginalizable).
These are CPU-bound (per-query solves); fine to run, just not GPU-accelerated.

In [ ]:
!python3 metaqa_confluence_2hop_ablation.py        # no-flow baseline (~0.88/0.94)
# !TRAIN_DEGNORM=1 python3 metaqa_confluence_2hop_v2.py   # flow model (slow: ~1hr+ on CPU)
# !python3 metaqa_3hop_pilot.py                     # 3-hop flow-vs-no-flow pilot

## 8. Regenerate the paper figures

In [ ]:
!python3 make_fig_coupling.py
!python3 make_fig_moe.py
!python3 make_fig_implicit.py
from IPython.display import Image, display
for f in ['paper/figs/coupling.png','paper/figs/moe_frontier.png','paper/figs/implicit_diff.png']:
    display(Image(f))

## 9. (Optional) commit new results back to the repo
Set your name/email, add a GitHub token remote, then push. Keeps commits under your name.
```python
!git config user.name 'Tanush Appapogu'
!git config user.email 'tanush.appapogu@gmail.com'
# !git remote set-url origin https://<YOUR_GITHUB_TOKEN>@github.com/tanushappapogu-max/Confluence.git
# !git add results paper && git commit -m 'Colab run results' && git push origin main
```